# QUICK FIX: добавляем только row_pos_in_req + точный блендинг

**Используй этот ноутбук если v2 уже работает (OOF 0.916, LB 91.79) и не хочешь переобучать всё с нуля.**

Этот ноутбук:
1. Перезагружает данные с правильным захватом `row_pos_in_req` (главная утечка!)
2. Переобучает только LGB и XGB (быстро — ~30 минут вместе)
3. Использует существующий CatBoost из v2 (можно сохранить test_pred_cb и oof_cb)
4. Точный bленд через scipy

**Время:** ~40 минут вместо 8-12 часов на полный v3.

**Ожидаемый эффект:** +0.1-0.3 п.п. (только от row_pos)


In [ ]:
!pip install -q lightgbm xgboost pyarrow

: 

In [37]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from scipy.optimize import minimize
import gc, os, warnings, time, pickle
warnings.filterwarnings('ignore')

DATA_DIR = './data'
SEED = 42
N_SPLITS = 5

## Шаг 1: Сохрани результаты CatBoost из v2

В **старом** ноутбуке (где CatBoost уже посчитан) выполни это, прежде чем закрывать:

```python
# В СТАРОМ ноутбуке (v2):
np.save('oof_cb_v2.npy', oof_cb)
np.save('test_pred_cb_v2.npy', test_pred_cb)
# Также сохрани id-порядок train и test чтобы корректно сопоставить!
train[['request_id', 'app_id', '__original_idx' if '__original_idx' in train.columns else 'request_id']].to_parquet('train_ids_v2.pq')
test[['request_id', 'variant_no', 'app_id']].to_parquet('test_ids_v2.pq')
```

**Затем скачай эти 4 файла локально** и загрузи в новый Colab или новую ячейку.

Если уже закрыл — придётся переобучить CatBoost (см. v3 ноутбук).


In [26]:
# === Загрузка данных и СРАЗУ row_pos_in_req ===
train = pd.read_parquet(f'{DATA_DIR}/train_dataset_small.pq')
test  = pd.read_parquet(f'{DATA_DIR}/test_dataset_small.pq')
feats = pd.read_parquet(f'{DATA_DIR}/features_small.pq')

# КЛЮЧЕВОЕ: сохраняем исходный порядок ДО любых merge
train['__original_idx'] = np.arange(len(train), dtype='int32')
test['__original_idx']  = np.arange(len(test), dtype='int32')

# row_pos_in_req по исходному порядку
for df in (train, test):
    df['row_pos_in_req'] = df.groupby('request_id', sort=False)['__original_idx'].rank(method='first').astype('int32') - 1
    g_size = df.groupby('request_id', sort=False)['request_id'].transform('size')
    df['row_pos_pct'] = (df['row_pos_in_req'] / g_size).astype('float32')
    df['row_from_end'] = (g_size - 1 - df['row_pos_in_req']).astype('int32')
    df['n_variants_in_req'] = g_size.astype('int32')

print('Sanity:')
print(train.groupby('row_pos_in_req')['is_deal'].mean().head(10).round(4))
print('Должно быть ~0.096 для pos=0 и убывать. Если так — row_pos работает.')

Sanity:
row_pos_in_req
0    0.0949
1    0.0778
2    0.0676
3    0.0608
4    0.0561
5    0.0506
6    0.0515
7    0.0483
8    0.0456
9    0.0463
Name: is_deal, dtype: float64
Должно быть ~0.096 для pos=0 и убывать. Если так — row_pos работает.


In [27]:
# === ВЕСЬ остальной препроцессинг как в v2 ===
NUMERIC_FROM_STR = ['rate', 'eva', 'eva_perc', 'ncl']
for df in (train, test):
    for col in NUMERIC_FROM_STR:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

CAT_STR = ['verif_compl', 'offer_type', 'risk_level_map',
           'verif_need', 'need_2ndfl', 'channel', 'basket_name']
for col in CAT_STR:
    all_vals = pd.concat([train[col], test[col]], axis=0).fillna('NA').astype(str)
    cats = pd.Categorical(all_vals).categories
    mp = {v: i for i, v in enumerate(cats)}
    train[col] = train[col].fillna('NA').astype(str).map(mp).astype('int32')
    test[col]  = test[col].fillna('NA').astype(str).map(mp).astype('int32')

# Features preprocessing
feats = feats.drop_duplicates(subset=['app_id', 'date_part'], keep='last').reset_index(drop=True)
for col in feats.select_dtypes(include=['object']).columns.tolist():
    if col in ('app_id', 'date_part'): continue
    converted = pd.to_numeric(feats[col], errors='coerce')
    if converted.notna().sum() / max(feats[col].notna().sum(), 1) > 0.9:
        feats[col] = converted.astype('float32')
    else:
        feats[col] = pd.Categorical(feats[col].fillna('NA').astype(str)).codes.astype('int32')
for col in feats.select_dtypes(include=['object']).columns.tolist():
    if col in ('app_id', 'date_part'): continue
    feats[col] = pd.Categorical(feats[col].fillna('NA').astype(str)).codes.astype('int32')

# Join
JOIN_KEYS = ['app_id', 'date_part']
feats_to_join = feats.drop(columns=[c for c in feats.columns if c in train.columns and c not in JOIN_KEYS], errors='ignore')
train = train.merge(feats_to_join, on=JOIN_KEYS, how='left')
test  = test.merge(feats_to_join,  on=JOIN_KEYS, how='left')

# Восстанавливаем порядок
train = train.sort_values('__original_idx').reset_index(drop=True)
test  = test.sort_values('__original_idx').reset_index(drop=True)

print('train:', train.shape, '| test:', test.shape)

train: (845110, 306) | test: (936883, 305)


In [28]:
# === Listwise FE ===
LISTWISE_BASE = ['rate', 'eva', 'eva_perc', 'ncl', 'limit', 'term']

def add_listwise_features(df, group_col='request_id'):
    g = df.groupby(group_col, sort=False)
    for col in LISTWISE_BASE:
        df[f'{col}_rank']    = g[col].rank(method='dense', pct=True).astype('float32')
        df[f'{col}_z']       = ((df[col] - g[col].transform('mean')) /
                                 g[col].transform('std').replace(0, 1)).astype('float32')
        g_min = g[col].transform('min')
        g_max = g[col].transform('max')
        g_med = g[col].transform('median')
        df[f'{col}_minus_min']    = (df[col] - g_min).astype('float32')
        df[f'{col}_minus_max']    = (df[col] - g_max).astype('float32')
        df[f'{col}_minus_median'] = (df[col] - g_med).astype('float32')
        df[f'{col}_to_min']    = (df[col] / g_min.replace(0, np.nan)).astype('float32')
        df[f'{col}_to_max']    = (df[col] / g_max.replace(0, np.nan)).astype('float32')
        df[f'{col}_range']     = (g_max - g_min).astype('float32')
        df[f'{col}_std']       = g[col].transform('std').astype('float32')
        df[f'is_min_{col}'] = (df[col] == g_min).astype('int8')
        df[f'is_max_{col}'] = (df[col] == g_max).astype('int8')
    return df

train = add_listwise_features(train)
test  = add_listwise_features(test)

# Domain features
def add_domain_features(df):
    df['eva_minus_ncl'] = (df['eva'] - df['ncl']).astype('float32')
    df['eva_div_ncl']   = (df['eva'] / (df['ncl'].abs() + 1e-9)).astype('float32')
    rate_m = df['rate'] / 12 / 100
    n = df['term'].astype('float32')
    df['monthly_payment'] = (df['limit'] * rate_m / (1 - (1 + rate_m).pow(-n) + 1e-9)).astype('float32')
    df['total_interest']  = (df['monthly_payment'] * n - df['limit']).astype('float32')
    df['eva_div_limit'] = (df['eva'] / (df['limit'] + 1e-9)).astype('float32')
    df['limit_div_req']  = (df['limit'] / df['req_loan_amount'].replace(0, np.nan)).astype('float32')
    df['term_div_req']   = (df['term']  / df['req_term'].replace(0, np.nan)).astype('float32')
    df['limit_minus_req']= (df['limit'] - df['req_loan_amount']).astype('float32')
    df['term_minus_req'] = (df['term'] - df['req_term']).astype('float32')
    df['rate_times_term'] = (df['rate'] * df['term']).astype('float32')
    if 'profit_income_out_rur_amt_cm' in df.columns:
        df['payment_to_income'] = (df['monthly_payment'] / (df['profit_income_out_rur_amt_cm'].abs() + 1e-9)).astype('float32')
    if 'age' in df.columns:
        df['term_to_age_left'] = (df['term'] / (12 * (70 - df['age']).clip(1, 70))).astype('float32')
    return df

train = add_domain_features(train)
test  = add_domain_features(test)

for col in ['eva_minus_ncl', 'monthly_payment', 'eva_div_limit', 'limit_div_req', 'rate_times_term']:
    for df in (train, test):
        df[f'{col}_rank'] = df.groupby('request_id', sort=False)[col].rank(method='dense', pct=True).astype('float32')

print('После FE:', train.shape, test.shape)
gc.collect()

После FE: (845110, 389) (936883, 388)


3

In [29]:
# === Count + Target Encoding ===
COUNT_COLS = ['variant_no', 'basket_name', 'offer_type', 'risk_level_map', 'offer_id']
all_df = pd.concat([train[COUNT_COLS], test[COUNT_COLS]], axis=0)
for col in COUNT_COLS:
    cnt_map = all_df[col].value_counts().to_dict()
    train[f'{col}_cnt'] = train[col].map(cnt_map).astype('float32')
    test[f'{col}_cnt']  = test[col].map(cnt_map).astype('float32')
for c1, c2 in [('variant_no', 'offer_type'), ('variant_no', 'risk_level_map')]:
    key_train = train[c1].astype(str) + '_' + train[c2].astype(str)
    key_test  = test[c1].astype(str)  + '_' + test[c2].astype(str)
    cnt = pd.concat([key_train, key_test], axis=0).value_counts().to_dict()
    train[f'{c1}_{c2}_cnt'] = key_train.map(cnt).astype('float32')
    test[f'{c1}_{c2}_cnt']  = key_test.map(cnt).astype('float32')
del all_df; gc.collect()

# OOF TE
train = train.sort_values('request_id').reset_index(drop=True)
test  = test.sort_values('request_id').reset_index(drop=True)
g_train_arr = train['request_id'].values
y_train_arr = train['is_deal'].values
GLOBAL_MEAN = float(y_train_arr.mean())
SMOOTHING = 20.0

TE_COLS = ['variant_no', 'basket_name', 'offer_type', 'risk_level_map', 'offer_id']
TE_CROSSES = []
if 'clientsegment' in train.columns:
    train['variant_x_segment'] = train['variant_no'].astype(str) + '_' + train['clientsegment'].astype(str)
    test['variant_x_segment']  = test['variant_no'].astype(str)  + '_' + test['clientsegment'].astype(str)
    TE_CROSSES.append('variant_x_segment')
train['variant_x_offer_type'] = train['variant_no'].astype(str) + '_' + train['offer_type'].astype(str)
test['variant_x_offer_type']  = test['variant_no'].astype(str)  + '_' + test['offer_type'].astype(str)
TE_CROSSES.append('variant_x_offer_type')

def oof_target_encode(train_df, test_df, col, y, groups, smoothing=20.0):
    oof = np.full(len(train_df), GLOBAL_MEAN, dtype='float32')
    gkf = GroupKFold(n_splits=5)
    for tr_idx, va_idx in gkf.split(train_df, y, groups=groups):
        stats_df = pd.DataFrame({col: train_df[col].iloc[tr_idx].values, 'y': y[tr_idx]})
        stats = stats_df.groupby(col)['y'].agg(['mean', 'count'])
        stats['te'] = (stats['mean'] * stats['count'] + GLOBAL_MEAN * smoothing) / (stats['count'] + smoothing)
        oof[va_idx] = train_df[col].iloc[va_idx].map(stats['te'].to_dict()).fillna(GLOBAL_MEAN).values
    full_stats_df = pd.DataFrame({col: train_df[col].values, 'y': y})
    full_stats = full_stats_df.groupby(col)['y'].agg(['mean', 'count'])
    full_stats['te'] = (full_stats['mean'] * full_stats['count'] + GLOBAL_MEAN * smoothing) / (full_stats['count'] + smoothing)
    test_enc = test_df[col].map(full_stats['te'].to_dict()).fillna(GLOBAL_MEAN).astype('float32').values
    return oof, test_enc

for col in TE_COLS + TE_CROSSES:
    print(f'TE: {col}')
    oof, te = oof_target_encode(train, test, col, y_train_arr, g_train_arr, SMOOTHING)
    train[f'{col}_te'] = oof
    test[f'{col}_te']  = te
for col in TE_CROSSES:
    train.drop(columns=[col], inplace=True)
    test.drop(columns=[col], inplace=True)
gc.collect()
print('Все фичи готовы')

TE: variant_no
TE: basket_name
TE: offer_type
TE: risk_level_map
TE: offer_id
TE: variant_x_segment
TE: variant_x_offer_type
Все фичи готовы


In [30]:
# === Финальные фичи ===
DROP_COLS = ['app_id', 'request_received', 'date_part', 'request_id',
             'offer_id', 'is_deal', 'nb_date', '__original_idx']

feature_cols = [c for c in train.columns
                if c not in DROP_COLS and c in test.columns]
print('Число фичей:', len(feature_cols))
print('row_pos_in_req in features?', 'row_pos_in_req' in feature_cols)

CAT_FEATURES = ['variant_no', 'verif_compl', 'offer_type', 'risk_level_map',
                'verif_need', 'need_2ndfl', 'channel', 'basket_name', 'pil1mtrx_offer']
CAT_FEATURES = [c for c in CAT_FEATURES if c in feature_cols]

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()
for c in CAT_FEATURES:
    X_train[c] = X_train[c].fillna(-1).astype('int32')
    X_test[c]  = X_test[c].fillna(-1).astype('int32')

print('X_train:', X_train.shape, 'X_test:', X_test.shape)

Число фичей: 395
row_pos_in_req in features? True
X_train: (845110, 395) X_test: (936883, 395)


In [31]:
def ndcg_at_k_grouped(df, score_col, target_col='is_deal', group_col='request_id', k=5):
    scores = []
    for req, g in df.groupby(group_col, sort=False):
        if g[target_col].sum() == 0:
            continue
        order = np.argsort(-g[score_col].values)
        ranks = np.where(g[target_col].values[order] == 1)[0]
        if len(ranks) == 0 or ranks[0] >= k:
            scores.append(0.0)
        else:
            scores.append(1.0 / np.log2(ranks[0] + 2))
    return np.mean(scores)

gkf = GroupKFold(n_splits=N_SPLITS)

## LightGBM (~10 минут на 5 фолдов)

In [32]:
oof_lgb = np.zeros(len(train))
test_pred_lgb = np.zeros(len(test))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train_arr, groups=g_train_arr)):
    print(f'\n===== LGB Fold {fold} =====')
    t0 = time.time()
    tr_groups = train.iloc[tr_idx].groupby('request_id', sort=False).size().values
    va_groups = train.iloc[va_idx].groupby('request_id', sort=False).size().values
    dtr = lgb.Dataset(X_train.iloc[tr_idx], y_train_arr[tr_idx], group=tr_groups, categorical_feature=CAT_FEATURES)
    dva = lgb.Dataset(X_train.iloc[va_idx], y_train_arr[va_idx], group=va_groups, categorical_feature=CAT_FEATURES, reference=dtr)
    params = {
        'objective': 'lambdarank', 'metric': 'ndcg', 'eval_at': [5],
        'lambdarank_truncation_level': 8, 'lambdarank_norm': True, 'label_gain': [0, 1],
        'learning_rate': 0.03, 'num_leaves': 127, 'min_data_in_leaf': 100,
        'feature_fraction': 0.8, 'feature_fraction_bynode': 0.8,
        'bagging_fraction': 0.8, 'bagging_freq': 5,
        'lambda_l1': 0.1, 'lambda_l2': 1.0, 'verbose': -1, 'seed': SEED + fold,
    }
    model = lgb.train(params, dtr, num_boost_round=5000, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(200), lgb.log_evaluation(200)])
    oof_lgb[va_idx] = model.predict(X_train.iloc[va_idx])
    test_pred_lgb += model.predict(X_test) / N_SPLITS
    print(f'Time: {(time.time() - t0)/60:.1f} мин')
    gc.collect()

train['oof_lgb'] = oof_lgb
print(f'\nOOF NDCG@5 (LGB): {ndcg_at_k_grouped(train, "oof_lgb"):.5f}')

# Сохраняем чтобы не потерять при крэше
np.save('oof_lgb_v3.npy', oof_lgb)
np.save('test_pred_lgb_v3.npy', test_pred_lgb)


===== LGB Fold 0 =====
Training until validation scores don't improve for 200 rounds
[200]	valid_0's ndcg@5: 0.91672
[400]	valid_0's ndcg@5: 0.916713
Early stopping, best iteration is:
[309]	valid_0's ndcg@5: 0.918337
Time: 2.4 мин

===== LGB Fold 1 =====
Training until validation scores don't improve for 200 rounds
[200]	valid_0's ndcg@5: 0.909591
Early stopping, best iteration is:
[190]	valid_0's ndcg@5: 0.909935
Time: 1.8 мин

===== LGB Fold 2 =====
Training until validation scores don't improve for 200 rounds
[200]	valid_0's ndcg@5: 0.914889
[400]	valid_0's ndcg@5: 0.913967
Early stopping, best iteration is:
[210]	valid_0's ndcg@5: 0.915215
Time: 1.9 мин

===== LGB Fold 3 =====
Training until validation scores don't improve for 200 rounds
[200]	valid_0's ndcg@5: 0.911912
[400]	valid_0's ndcg@5: 0.910329
Early stopping, best iteration is:
[203]	valid_0's ndcg@5: 0.912113
Time: 1.9 мин

===== LGB Fold 4 =====
Training until validation scores don't improve for 200 rounds
[200]	valid_

## XGBoost (~5 минут на 5 фолдов)

In [33]:
oof_xgb = np.zeros(len(train))
test_pred_xgb = np.zeros(len(test))
X_train_xgb = X_train.fillna(-999.0)
X_test_xgb  = X_test.fillna(-999.0)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train_xgb, y_train_arr, groups=g_train_arr)):
    print(f'\n===== XGB Fold {fold} =====')
    t0 = time.time()
    tr_groups = train.iloc[tr_idx].groupby('request_id', sort=False).size().values
    va_groups = train.iloc[va_idx].groupby('request_id', sort=False).size().values
    dtr = xgb.DMatrix(X_train_xgb.iloc[tr_idx], label=y_train_arr[tr_idx], group=tr_groups)
    dva = xgb.DMatrix(X_train_xgb.iloc[va_idx], label=y_train_arr[va_idx], group=va_groups)
    params = {
        'objective': 'rank:pairwise', 'eval_metric': 'ndcg@5',
        'learning_rate': 0.03, 'max_depth': 8, 'min_child_weight': 5,
        'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0,
        'tree_method': 'hist', 'device': 'cuda', 'seed': SEED + fold, 'verbosity': 0,
    }
    model_xgb = xgb.train(params, dtr, num_boost_round=5000, evals=[(dva, 'va')],
                           early_stopping_rounds=300, verbose_eval=200)
    oof_xgb[va_idx] = model_xgb.predict(dva)
    test_pred_xgb += model_xgb.predict(xgb.DMatrix(X_test_xgb)) / N_SPLITS
    print(f'Time: {(time.time() - t0)/60:.1f} мин')
    gc.collect()

train['oof_xgb'] = oof_xgb
print(f'\nOOF NDCG@5 (XGB): {ndcg_at_k_grouped(train, "oof_xgb"):.5f}')

np.save('oof_xgb_v3.npy', oof_xgb)
np.save('test_pred_xgb_v3.npy', test_pred_xgb)


===== XGB Fold 0 =====
[0]	va-ndcg@5:0.89083
[200]	va-ndcg@5:0.91630
[400]	va-ndcg@5:0.91907
[600]	va-ndcg@5:0.91908
[800]	va-ndcg@5:0.91892
[945]	va-ndcg@5:0.91862
Time: 0.9 мин

===== XGB Fold 1 =====
[0]	va-ndcg@5:0.87841
[200]	va-ndcg@5:0.90852
[400]	va-ndcg@5:0.91051
[600]	va-ndcg@5:0.91159
[800]	va-ndcg@5:0.91116
[939]	va-ndcg@5:0.91055
Time: 0.9 мин

===== XGB Fold 2 =====
[0]	va-ndcg@5:0.89053
[200]	va-ndcg@5:0.91395
[400]	va-ndcg@5:0.91569
[600]	va-ndcg@5:0.91592
[800]	va-ndcg@5:0.91592
[815]	va-ndcg@5:0.91602
Time: 0.8 мин

===== XGB Fold 3 =====
[0]	va-ndcg@5:0.88603
[200]	va-ndcg@5:0.91099
[400]	va-ndcg@5:0.91308
[600]	va-ndcg@5:0.91366
[800]	va-ndcg@5:0.91317
[964]	va-ndcg@5:0.91265
Time: 0.9 мин

===== XGB Fold 4 =====
[0]	va-ndcg@5:0.88062
[200]	va-ndcg@5:0.91463
[400]	va-ndcg@5:0.91540
[600]	va-ndcg@5:0.91693
[800]	va-ndcg@5:0.91594
[875]	va-ndcg@5:0.91611
Time: 0.8 мин

OOF NDCG@5 (XGB): 0.91479


## Загрузить CatBoost из v2 (если есть сохранённые .npy файлы)

Если ты сохранил oof_cb_v2.npy и test_pred_cb_v2.npy из старого запуска — загрузи их.
Иначе придётся пересчитать (см. v3 ноутбук).


In [34]:
# Если есть сохранённые CB из v2 — используем
import os

if os.path.exists('oof_cb_v2.npy') and os.path.exists('test_pred_cb_v2.npy'):
    oof_cb = np.load('oof_cb_v2.npy')
    test_pred_cb = np.load('test_pred_cb_v2.npy')
    print('CB из v2 загружен:', oof_cb.shape, test_pred_cb.shape)
    
    # ВАЖНО: проверь что порядок строк совпадает!
    # В v2 train сортировался по request_id перед обучением, здесь тоже.
    # Должно совпасть. Если нет — пересчитай CB.
    
    train['oof_cb'] = oof_cb
    print(f'OOF NDCG@5 (CB из v2): {ndcg_at_k_grouped(train, "oof_cb"):.5f}')
else:
    print('Файлы oof_cb_v2.npy / test_pred_cb_v2.npy не найдены.')
    print('Либо: 1) пересчитать CatBoost (см. v3 ноутбук)')
    print('Либо: 2) сделать blend только из LGB + XGB')
    oof_cb = None
    test_pred_cb = None

CB из v2 загружен: (845110,) (936883,)
OOF NDCG@5 (CB из v2): 0.19198


## Финальный blend через scipy.optimize

In [35]:
def rank_within_group(df, score_col, group_col='request_id'):
    return df.groupby(group_col, sort=False)[score_col].rank(method='average', pct=True).values

train['r_lgb'] = rank_within_group(train, 'oof_lgb')
train['r_xgb'] = rank_within_group(train, 'oof_xgb')

if oof_cb is not None:
    train['r_cb'] = rank_within_group(train, 'oof_cb')
    
    print('Grid search:')
    best_grid_w, best_grid_s = (0.33, 0.34, 0.33), 0
    for w_lgb in np.arange(0, 1.01, 0.05):
        for w_cb in np.arange(0, 1.01 - w_lgb, 0.05):
            w_xgb = 1.0 - w_lgb - w_cb
            if w_xgb < 0: continue
            train['blend'] = w_lgb * train['r_lgb'] + w_cb * train['r_cb'] + w_xgb * train['r_xgb']
            s = ndcg_at_k_grouped(train, 'blend')
            if s > best_grid_s:
                best_grid_s, best_grid_w = s, (w_lgb, w_cb, w_xgb)
    print(f'Grid: {best_grid_w}, OOF={best_grid_s:.5f}')

    def neg_ndcg(w):
        w = np.abs(w); w = w / w.sum()
        train['blend'] = w[0] * train['r_lgb'] + w[1] * train['r_cb'] + w[2] * train['r_xgb']
        return -ndcg_at_k_grouped(train, 'blend')
    
    res = minimize(neg_ndcg, x0=best_grid_w, method='Nelder-Mead',
                   options={'xatol': 0.001, 'fatol': 1e-7, 'maxiter': 500})
    best_w = np.abs(res.x); best_w = tuple(best_w / best_w.sum())
    print(f'Nelder-Mead: {[round(w, 4) for w in best_w]}, OOF={-res.fun:.5f}')
    print(f'Балл: {-res.fun * 100:.4f}')
else:
    # Только LGB + XGB
    print('Blend только LGB + XGB:')
    best_score, best_w = 0, (0.5, 0.5)
    for w_lgb in np.arange(0, 1.01, 0.02):
        w_xgb = 1.0 - w_lgb
        train['blend'] = w_lgb * train['r_lgb'] + w_xgb * train['r_xgb']
        s = ndcg_at_k_grouped(train, 'blend')
        if s > best_score: best_score, best_w = s, (w_lgb, w_xgb)
    print(f'LGB/XGB: {best_w}, OOF={best_score:.5f}')

Grid search:
Grid: (np.float64(0.4), np.float64(0.0), np.float64(0.6)), OOF=0.91507


NameError: name 'minimize' is not defined

In [ ]:
def neg_ndcg(w):
    w = np.abs(w); w = w / w.sum()
    train['blend'] = w[0] * train['r_lgb'] + w[1] * train['r_cb'] + w[2] * train['r_xgb']
    return -ndcg_at_k_grouped(train, 'blend')

res = minimize(neg_ndcg, x0=best_grid_w, method='Nelder-Mead',
                options={'xatol': 0.001, 'fatol': 1e-7, 'maxiter': 500})
best_w = np.abs(res.x); best_w = tuple(best_w / best_w.sum())
print(f'Nelder-Mead: {[round(w, 4) for w in best_w]}, OOF={-res.fun:.5f}')
print(f'Балл: {-res.fun * 100:.4f}')

In [ ]:
# === Submission ===
test['pred_lgb'] = test_pred_lgb
test['pred_xgb'] = test_pred_xgb
test['r_lgb'] = rank_within_group(test, 'pred_lgb')
test['r_xgb'] = rank_within_group(test, 'pred_xgb')

if oof_cb is not None and test_pred_cb is not None:
    test['pred_cb'] = test_pred_cb
    test['r_cb'] = rank_within_group(test, 'pred_cb')
    test['score'] = best_w[0] * test['r_lgb'] + best_w[1] * test['r_cb'] + best_w[2] * test['r_xgb']
else:
    test['score'] = best_w[0] * test['r_lgb'] + best_w[1] * test['r_xgb']

submission = test[['request_id', 'variant_no', 'score']].copy()
submission.to_csv('submission_quickfix.csv', index=False, sep=';')
print('shape:', submission.shape)
print(submission.head(5))
assert submission['score'].isna().sum() == 0
print('OK')

In [ ]:
# Скачать
try:
    from google.colab import files
    files.download('submission_quickfix.csv')
except:
    import subprocess
    out = subprocess.run(['curl', '-s', '-F', 'file=@submission_quickfix.csv', 'https://0x0.st'],
                          capture_output=True, text=True)
    print('Ссылка:', out.stdout.strip())